# SkillScope: Job Market & Skill-Gap Intelligence Warehouse & Analytics
**National University of Computer & Emerging Sciences (FAST-NUCES), Karachi Campus**  
**Section:** BDS-5B  
**Courses:** Data Warehousing & Business Intelligence (DWBI) + Data Analytics & Visualization (DAV)  
**Group Members:**
- `24K-2505` — Myer Kamran
- `24K-2521` — Shazil Ahmed
- `24K-2583` — Ali John Rajani

---

In [1]:
import os, sys, json
import pandas as pd
import numpy as np

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

SILVER_DIR = os.path.join(PROJECT_ROOT, 'data', 'silver')
GOLD_DIR = os.path.join(PROJECT_ROOT, 'data', 'gold')
print('Silver Layer:', SILVER_DIR)
print('Gold Layer:', GOLD_DIR)

## 1. Lakehouse Layer Reconciliation & Data Quality Gate
Inspects Bronze $\rightarrow$ Silver $\rightarrow$ Gold row counts, PK uniqueness, FK referential integrity, and schema summary across all 11 Dimensions and 5 Constellation Fact/Bridge tables.

In [2]:
with open(os.path.join(GOLD_DIR, 'etl_quality_reconciliation.json'), 'r') as f:
    recon = json.load(f)
print(f"Quality Gate Status: {recon['overall_status']} ({recon['checks_passed']}/{recon['checks_total']} checks passed)")
print('Bronze counts:', recon['bronze_row_counts'])
print('Silver counts:', recon['silver_row_counts'])

tables = [
    'dim_industry', 'dim_country', 'dim_role_family', 'dim_skill_category',
    'dim_date', 'dim_company', 'dim_location', 'dim_role', 'dim_seniority', 'dim_source', 'dim_skill',
    'fact_job_posting', 'fact_posting_skill', 'fact_survey_response', 'fact_survey_skill', 'fact_skill_gap_result'
]
summary_rows = []
for t in tables:
    df_t = pd.read_parquet(os.path.join(GOLD_DIR, f'{t}.parquet'))
    summary_rows.append({'Table Name': t, 'Row Count': len(df_t), 'Columns': len(df_t.columns)})
pd.DataFrame(summary_rows)

## 2. Independent 100-Posting NLP Skill Extraction Benchmark
Evaluates Precision, Recall, F1-Score, and Hard-Negative Specificity of the ESCO-aligned rule-based extractor against `nlp/ground_truth_100.json` (75 tech/data postings + 25 non-tech hard negatives), including False Positive / False Negative error analysis.

In [3]:
with open(os.path.join(GOLD_DIR, 'nlp_evaluation_report.json'), 'r') as f:
    nlp_eval = json.load(f)
print(f"Mode: {nlp_eval['evaluation_mode']} | Sample Size: {nlp_eval['sample_size']} postings")
print(f"TP: {nlp_eval['true_positives']} | FP: {nlp_eval['false_positives']} | FN: {nlp_eval['false_negatives']}")
print(f"Micro Precision: {nlp_eval['precision']*100:.2f}% | Recall: {nlp_eval['recall']*100:.2f}% | F1-Score: {nlp_eval['f1_score']*100:.2f}%")
print(f"Hard-Negative Clean Rate (25 non-tech postings): {nlp_eval['hard_negative_clean_rate']*100:.1f}%")
print('Sample False Positives:', nlp_eval['false_positive_examples'][:3])
print('Sample False Negatives:', nlp_eval['false_negative_examples'][:3])
pd.DataFrame(nlp_eval['samples']).head(10)

## 3. Exploratory Data Analysis (EDA): Authentic Wave Shifts, Co-Occurrence & 5-Fold CV Salary Influencers

In [4]:
with open(os.path.join(GOLD_DIR, 'eda_summary_report.json'), 'r') as f:
    eda = json.load(f)
print('KPIs:', json.dumps(eda['kpis'], indent=2))

df_yoy = pd.read_parquet(os.path.join(GOLD_DIR, 'gold_skill_yoy_growth.parquet'))
print('=== Rising Skills (Wave 2 Oct-Nov 2023 vs Wave 1 Jun-Sep 2023 Normalized Share %) ===')
display(df_yoy[df_yoy['total_mentions'] >= 20].head(10)[['skill_name', 'category_name', 'mentions_2023', 'mentions_2024', 'wave1_share_pct', 'wave2_share_pct', 'yoy_growth_pct']])

df_cooc = pd.read_parquet(os.path.join(GOLD_DIR, 'gold_skill_cooccurrence.parquet'))
print('=== Top 10 Co-Occurring Skill Pairs ===')
display(df_cooc.head(10))

df_inf = pd.read_parquet(os.path.join(GOLD_DIR, 'gold_salary_key_influencers.parquet'))
print('=== Top 12 Salary Key Influencers (5-Fold CV Random Forest) ===')
display(df_inf.head(12))

## 4. Knowledge-Based Expert System with Mamdani Fuzzy Priority Scoring
Demonstrates live skill-gap recommendation with Mamdani explainability for a FAST-NUCES BS Data Science student targeting a **Data Scientist** role.

In [5]:
from ai.fuzzy_recommender import FuzzySkillGapRecommender

recommender = FuzzySkillGapRecommender(
    role_skill_profiles_df=pd.read_parquet(os.path.join(GOLD_DIR, 'gold_role_skill_profiles.parquet')),
    cooccurrence_df=pd.read_parquet(os.path.join(GOLD_DIR, 'gold_skill_cooccurrence.parquet')),
    skills_df=pd.read_parquet(os.path.join(GOLD_DIR, 'dim_skill.parquet')),
    roles_df=pd.read_parquet(os.path.join(GOLD_DIR, 'dim_role.parquet')),
)

# Known skills: Python (1), SQL (2), Pandas & NumPy (24), Excel (30)
match_score, rec_df = recommender.recommend_for_profile(
    profile_id='FAST_BDS_5B',
    profile_label='FAST-NUCES BDS-5B Student (Python, SQL, Pandas, Excel)',
    role_key=1,
    known_skill_keys=[1, 2, 24, 30],
    top_n=10
)
print(f'Target Role: Data Scientist | Current Role Match Score: {match_score:.2f}%')
rec_df.merge(pd.read_parquet(os.path.join(GOLD_DIR, 'dim_skill.parquet'))[['skill_key', 'skill_name']], on='skill_key')[[
    'priority_rank', 'skill_name', 'fuzzy_priority_level', 'gap_score', 'demand_pct', 'salary_premium_pct', 'rationale'
]]